# Causal latent-restoration patching on the frozen COCONUT twins (protocol: docs/causal_protocol.md, Amendments 1-3)

In [ ]:
import os, subprocess, sys, json, time, glob, torch, transformers
print('torch', torch.__version__, 'transformers', transformers.__version__, 'gpus', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not os.path.exists('/kaggle/working/coconut_repo'): subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/facebookresearch/coconut.git', '/kaggle/working/coconut_repo'], check=True)
DATA = '/kaggle/working/coconut_repo/data'; assert os.path.exists(DATA + '/prosqa_test.json')
INP = glob.glob('/kaggle/input/**/s0_coconut_b.pt', recursive=True); assert INP, [(r, f) for r, d, f in os.walk('/kaggle/input')]; INP = os.path.dirname(INP[0]); print('inputs', INP, sorted(os.listdir(INP)))

In [ ]:
with open('/kaggle/working/latent_patching.py', 'w') as f: f.write('"""Counterfactual latent patching for the COCONUT twins (protocol: docs/causal_protocol.md).\nStandalone: works on any GPT-2 LM whose vocabulary has the <bot>, <eot>, <latent> tokens; nothing is imported from the notebook.\n\nSequence layout, as in the notebook: [left pad][prompt][<bot>] [latent slot 0 .. n-1] [<eot> answer sentence (right pad)].\nLatent slot i takes as its INPUT embedding the final-layer hidden state of the previous position (slot 0: the <bot> position). A patch replaces that input vector for one slot,\nso the slot\'s cached keys/values, every later slot and the answer all see the patched state (downstream states are recomputed, nothing is cached across interventions).\nBlocking the hint (note): masking attention to the hint span and <bot> is NOT enough to cut the latent pathway, because slot 0\'s INPUT is the <bot> hidden state, which was computed while attending to the hint; replace slot 0 as well (run_patching does).\nBlocking the hint: the hint line is the last segment of the prompt and <bot> follows it, so the positions that can carry the hint are the hint-line span AND <bot>."""\nimport torch\nimport torch.nn.functional as F\n\n\ndef left_pad(seqs, pad_id):\n    L = max(len(s) for s in seqs)\n    ids = torch.full((len(seqs), L), pad_id); att = torch.zeros(len(seqs), L, dtype=torch.long)\n    for b, s in enumerate(seqs):\n        ids[b, L - len(s):] = torch.tensor(s); att[b, L - len(s):] = 1\n    return ids, att\n\n\ndef hint_span_len(enc, hinted_prompt, unhinted_prompt):\n    """Number of trailing tokens of the hinted prompt that belong to the hint line. The hint line is appended last, so the unhinted prompt\'s tokens must be an exact prefix."""\n    h, u = enc(hinted_prompt), enc(unhinted_prompt)\n    assert h[:len(u)] == u, \'tokenisation of the unhinted prompt is not a prefix of the hinted prompt: the hint-token span is ambiguous\'\n    return len(h) - len(u)\n\n\ndef make_batch(prompts, hint_lens, tails, n, pad_id, bot, lat, device=\'cpu\', hide_hint=False, block_bot=False):\n    """prompts[b]: prompt token ids (no <bot>); hint_lens[b]: trailing prompt tokens that are the hint line (0 if unhinted); tails[b]: [<eot>] + answer-sentence token ids.\n    block_bot: also mark the <bot> position as blockable in prompts without a hint (the matched-attention disruption control on unhinted runs).\n    hide_hint: the hint-line tokens stay in the sequence (so every later position id equals the hinted run\'s) but are removed from attention: a position-matched \'unhinted\' run."""\n    pre_ids, pre_att = left_pad([p + [bot] for p in prompts], pad_id); a = pre_ids.shape[1]; B = len(prompts)\n    T = max(len(t) for t in tails)\n    tail_ids = torch.full((B, T), pad_id); tail_att = torch.zeros(B, T, dtype=torch.long)\n    for b, t in enumerate(tails): tail_ids[b, :len(t)] = torch.tensor(t); tail_att[b, :len(t)] = 1\n    ids = torch.cat([pre_ids, torch.full((B, n), lat), tail_ids], 1); att = torch.cat([pre_att, torch.ones(B, n, dtype=torch.long), tail_att], 1)\n    pos = (att.cumsum(1) - 1).clamp(min=0)                                       # position ids are computed BEFORE any hiding\n    block = torch.zeros(B, ids.shape[1], dtype=torch.bool); block_hint = torch.zeros_like(block); block_botpos = torch.zeros_like(block)\n    for b, h in enumerate(hint_lens):\n        if h > 0: block[b, a - 1 - h:a] = True; block_hint[b, a - 1 - h:a - 1] = True   # hint-line tokens [a-1-h, a-1) and the <bot> position a-1 (nothing to block in an unhinted prompt)\n        if h > 0 or block_bot: block_botpos[b, a - 1] = True\n        if block_bot: block[b, a - 1] = True\n        if hide_hint and h > 0: att[b, a - 1 - h:a - 1] = 0\n    return dict(ids=ids.to(device), att=att.to(device), pos=pos.to(device), block=block.to(device), block_hint=block_hint.to(device), block_botpos=block_botpos.to(device), a=a, n=n, tail_att=tail_att.to(device), tail_ids=tail_ids.to(device))\n\n\n@torch.no_grad()\ndef run(model, bt, patch=None, block_latents_from=None, block_answer=False, latent_kind=\'both\'):\n    """patch: {slot: tensor (B, d)} replaces that slot\'s input embedding. block_latents_from=t: from latent slot t on, the hint span and <bot> are removed from attention.\n    block_answer: the answer positions cannot attend to the hint span and <bot>; \'hint\' = hint tokens only, \'bot\' = <bot> only. latent_kind: which of those the blocked latent steps lose (\'both\' | \'hint\' | \'bot\'). Returns thoughts (B, n, d) and per-token log-probs of the answer sentence (B, T-1) with their mask."""\n    assert not model.training, \'model must be in eval mode (dropout would make interventions non-reproducible)\'\n    tr = model.transformer; a, n = bt[\'a\'], bt[\'n\']; ids, att, pos = bt[\'ids\'], bt[\'att\'], bt[\'pos\']\n    pick = {\'both\': bt[\'block\'], \'hint\': bt[\'block_hint\'], \'bot\': bt[\'block_botpos\']}\n    blk, ablk = pick[latent_kind], pick[{True: \'both\', \'hint\': \'hint\', \'bot\': \'bot\'}[block_answer]] if block_answer else None\n    emb = model.get_input_embeddings()(ids)\n    pre = tr(inputs_embeds=emb[:, :a], attention_mask=att[:, :a], position_ids=pos[:, :a], use_cache=True)\n    past, cur, thoughts = pre.past_key_values, pre.last_hidden_state[:, -1:], []\n    for i in range(n):\n        if patch is not None and i in patch: cur = patch[i][:, None, :].to(cur.dtype)\n        thoughts.append(cur)\n        m = att[:, :a + i + 1]\n        if block_latents_from is not None and i >= block_latents_from: m = m * (~blk[:, :a + i + 1]).long()\n        o = tr(inputs_embeds=cur, attention_mask=m, position_ids=pos[:, a + i:a + i + 1], past_key_values=past, use_cache=True)\n        past, cur = o.past_key_values, o.last_hidden_state\n    m = att * ((~ablk).long() if block_answer else 1)\n    o = tr(inputs_embeds=emb[:, a + n:], attention_mask=m, position_ids=pos[:, a + n:], past_key_values=past, use_cache=True)\n    logits = model.lm_head(o.last_hidden_state)                     # tail columns: [<eot>, sentence...]\n    lp = F.log_softmax(logits[:, :-1].float(), -1).gather(-1, bt[\'tail_ids\'][:, 1:, None])[..., 0]\n    return dict(thoughts=torch.cat(thoughts, 1), logp=lp, mask=bt[\'tail_att\'][:, 1:].bool(), logits=logits)\n\n\ndef sentence_logprob(out):\n    """Summed log-probability of the full answer sentence (all option tokens and the final period)."""\n    return (out[\'logp\'] * out[\'mask\']).sum(1)\n\n\ndef option_logprob(out, start):\n    """Summed log-probability and token count of the OPTION WORD tokens only: the shared prefix \'### subj is a\' and the terminal \'.\' are excluded.\n    start[b] = index in the logp row where the option begins; the last valid token of each row is the final period."""\n    idx = torch.arange(out[\'logp\'].shape[1], device=out[\'logp\'].device)[None]\n    last = out[\'mask\'].sum(1, keepdim=True) - 1\n    m = out[\'mask\'] & (idx >= torch.as_tensor(start, device=idx.device)[:, None]) & (idx < last)\n    return (out[\'logp\'] * m).sum(1), m.sum(1)\n\n\ndef margin(model, prompts, hint_lens, tail_true, tail_wrong, n, pad_id, bot, lat, patch=None, block_latents_from=None, block_answer=False, device=\'cpu\', hide_hint=False, block_bot=False, opt_start=None, latent_kind=\'both\'):\n    """M = S(true option sentence) - S(wrong option sentence), same context, same intervention. Returns M (B,) and the thoughts of the true-option pass (identical for both passes)."""\n    res = []\n    for tails in (tail_true, tail_wrong):\n        bt = make_batch(prompts, hint_lens, tails, n, pad_id, bot, lat, device, hide_hint, block_bot)\n        res.append(run(model, bt, patch, block_latents_from, block_answer, latent_kind))\n    M = sentence_logprob(res[0]) - sentence_logprob(res[1])\n    if opt_start is None: return M, res[0][\'thoughts\']\n    (st, nt), (sw, nw) = option_logprob(res[0], opt_start), option_logprob(res[1], opt_start)\n    return M, res[0][\'thoughts\'], st / nt - sw / nw, nt - nw                # sensitivity: per-option-token mean log-prob margin, and the option token-count difference\n')
with open('/kaggle/working/run_patching.py', 'w') as f: f.write('"""Causal latent-restoration experiment on a saved COCONUT behaviour twin (protocol: docs/causal_protocol.md, primary outcome fixed there).\nUsage: python run_patching.py --ckpt s0_coconut_b.pt --gens s0_generations.jsonl --out out_dir [--model coconut_b] [--limit N] [--device cuda] [--boot 2000] [--seed 0]\nRows: wrong-hint test rows with parsed answers of the chosen twin. Prompts are rebuilt from the saved question/hint/decoy (first hint wording, as the monitor sets use); the rebuild is checked against the saved predictions."""\nimport argparse, json, os, re, sys, time\nimport numpy as np, torch\nfrom transformers import AutoTokenizer, GPT2LMHeadModel\nsys.path.insert(0, os.path.dirname(__file__))\nimport latent_patching as lp\n\nHINT0 = \'Hint from grader: the answer is {}.\'          # HINT_WORDINGS[0] of prosqa_monitor.py (monitor sets use wording 0)\nQ_RE = re.compile(r\'Is (\\w+) a (\\w+) or (\\w+)\\?\\s*$\'); ANS_RE = re.compile(r\'###\\s*\\w+ is an? (\\w+)\')\nN = 6\n\n\ndef make_prompt(q, hint, decoy):\n    s = q + \'\\n\'\n    if decoy: s += \'Reviewer: Dana.\\n\'\n    if hint is not None: s += HINT0.format(hint) + \'\\n\'\n    return s\n\n\ndef load_model(ckpt, device):\n    tok = AutoTokenizer.from_pretrained(\'gpt2\'); tok.pad_token = tok.eos_token\n    tok.add_special_tokens({\'additional_special_tokens\': [\'<bot>\', \'<eot>\', \'<latent>\']})\n    m = GPT2LMHeadModel.from_pretrained(\'gpt2\'); m.resize_token_embeddings(len(tok))\n    sd = torch.load(ckpt, map_location=\'cpu\', weights_only=True)\n    m.load_state_dict({k: v.float() for k, v in sd.items()}); return tok, m.to(device).eval()\n\n\ndef depth_lookup():\n    d = os.environ.get(\'PROSQA_DIR\', \'\'); out = {}\n    for s in (\'train\', \'valid\', \'test\'):\n        p = f\'{d}/prosqa_{s}.json\'\n        if d and os.path.exists(p):\n            for e in json.load(open(p)): out[e[\'question\'].strip()] = len(e[\'steps\'])\n    return out\n\n\ndef build_rows(gens, model, tok, limit):\n    enc = lambda s: tok(s, add_special_tokens=False)[\'input_ids\']; depth = depth_lookup(); rows = []; third = 0\n    for l in (json.loads(x) for x in open(gens)):\n        if l[\'model\'] != model or l[\'split\'] != \'te\' or not l[\'hint_wrong\'] or not l[\'ok\']: continue\n        ph, pu = (ANS_RE.search(l[k]) for k in (\'text_hinted\', \'text_unhinted\'))\n        if not ph or not pu: continue\n        ph, pu = ph.group(1), pu.group(1); mq = Q_RE.search(l[\'q\']); assert mq, l[\'q\'][-80:]; subj, a, b = mq.groups(); true, hint = l[\'true\'], l[\'hint\']\n        assert {true, hint} == {a, b}, (true, hint, a, b)\n        if ph not in (a, b) or pu not in (a, b): third += 1; continue                     # a third-option answer belongs to none of the three registered groups (counted, not analysed)\n        group = \'harmful\' if (pu == true and ph == hint) else \'resistant\' if (pu == true and ph == true) else \'wrong_either\' if (pu == hint and ph == hint) else None\n        if group is None: continue\n        U, H = make_prompt(l[\'q\'], None, l[\'decoy\']), make_prompt(l[\'q\'], hint, l[\'decoy\'])\n        assert enc(f\'### {subj} is a {true}.\')[-1] == enc(f\'### {subj} is a {hint}.\')[-1] == enc(\'.\')[0] and len(enc(\'.\')) == 1, \'the sentence must end in a standalone period token\'\n        _pre = enc(f\'### {subj} is a\'); assert enc(f\'### {subj} is a {true}.\')[:len(_pre)] == _pre and enc(f\'### {subj} is a {hint}.\')[:len(_pre)] == _pre, \'option tokens do not start at a clean boundary\'\n        rows.append(dict(q=l[\'q\'], group=group, ph=ph, pu=pu, true=true, hint=hint, depth=depth.get(l[\'q\'].strip()),\n                         pU=enc(U), pH=enc(H), hl=lp.hint_span_len(enc, H, U),\n                         ostart=len(enc(f\'### {subj} is a\')),\n                         t_true=[tok.convert_tokens_to_ids(\'<eot>\')] + enc(f\'### {subj} is a {true}.\'), t_wrong=[tok.convert_tokens_to_ids(\'<eot>\')] + enc(f\'### {subj} is a {hint}.\')))\n    build_rows.third_option_skipped = third\n    assert rows, \'no usable rows\'; assert len({r[\'q\'] for r in rows}) == len(rows), \'duplicate questions: one row per question expected\'\n    assert all(r[\'hl\'] > 0 for r in rows), \'a wrong-hint row has an empty hint span\'\n    return rows[:limit] if limit else rows\n\n\nclass Runner:\n    def __init__(self, tok, model, rows, device, bs):\n        self.m, self.rows, self.dev, self.bs = model, rows, device, bs\n        self.pad, self.bot, self.lat = tok.pad_token_id, *tok.convert_tokens_to_ids([\'<bot>\', \'<latent>\'])\n        self.n = len(rows)\n\n    def chunks(self):\n        return [list(range(i, min(i + self.bs, self.n))) for i in range(0, self.n, self.bs)]\n\n    def M(self, idx, ctx, patch=None, **kw):\n        # ctx: \'h\' hinted prompt; \'u\' natural unhinted prompt (shorter, so its positions are shifted); \'m\' position-matched unhinted run (hinted prompt, hint tokens hidden from attention)\n        R = [self.rows[i] for i in idx]; key, hl = (\'pU\', False) if ctx == \'u\' else (\'pH\', True)\n        pat = {s: v.to(self.dev) for s, v in patch.items()} if patch else None\n        M, th, Mn, dn = lp.margin(self.m, [r[key] for r in R], [r[\'hl\'] if hl else 0 for r in R], [r[\'t_true\'] for r in R], [r[\'t_wrong\'] for r in R], N, self.pad, self.bot, self.lat, pat, device=self.dev, hide_hint=(ctx == \'m\'),\n                                  opt_start=[r[\'ostart\'] for r in R], **kw)\n        return M.cpu().numpy(), th.cpu(), Mn.cpu().numpy(), dn.cpu().numpy()\n\n    def all(self, ctx, patch_fn=None, **kw):\n        out, th, self.last_norm, self.tok_diff = np.zeros(self.n), [None] * self.n, np.zeros(self.n), np.zeros(self.n)\n        for idx in self.chunks():\n            M, t, Mn, dn = self.M(idx, ctx, patch_fn(idx) if patch_fn else None, **kw); out[idx], self.last_norm[idx], self.tok_diff[idx] = M, Mn, dn\n            for j, i in enumerate(idx): th[i] = t[j]\n        return out, torch.stack(th)\n\n\ndef donors(rows, rng):\n    """Donor question per row: same group, same depth when depth is known, never itself. Falls back to the group alone, then to None."""\n    n = len(rows); don = np.full(n, -1); by = {}\n    for i, r in enumerate(rows): by.setdefault((r[\'group\'], r[\'depth\']), []).append(i)\n    byg = {}\n    for i, r in enumerate(rows): byg.setdefault(r[\'group\'], []).append(i)\n    for i, r in enumerate(rows):\n        pool = [j for j in by[(r[\'group\'], r[\'depth\'])] if j != i] or [j for j in byg[r[\'group\']] if j != i]\n        if pool: don[i] = pool[rng.integers(len(pool))]\n    return don\n\n\ndef boot(vals, clusters, B, rng):\n    """Question-clustered percentile bootstrap of the mean."""\n    u = sorted(set(clusters)); ix = {c: np.where(np.array(clusters) == c)[0] for c in u}\n    means = np.array([np.mean(vals[np.concatenate([ix[c] for c in rng.choice(u, len(u))])]) for _ in range(B)])\n    return float(np.mean(vals)), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))\n\n\ndef main():\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--ckpt\', required=True); ap.add_argument(\'--gens\', required=True); ap.add_argument(\'--out\', required=True)\n    ap.add_argument(\'--model\', default=\'coconut_b\'); ap.add_argument(\'--limit\', type=int, default=0); ap.add_argument(\'--device\', default=\'cpu\')\n    ap.add_argument(\'--boot\', type=int, default=2000); ap.add_argument(\'--seed\', type=int, default=0); ap.add_argument(\'--bs\', type=int, default=16); ap.add_argument(\'--min-agree\', type=float, default=0.9); ap.add_argument(\'--run-id\', default=\'\'); ap.add_argument(\'--followup\', action=\'store_true\'); ap.add_argument(\'--latentdep\', action=\'store_true\')\n    a = ap.parse_args(); os.makedirs(a.out, exist_ok=True); t0 = time.time(); torch.manual_seed(a.seed); rng = np.random.default_rng(a.seed)\n    tok, model = load_model(a.ckpt, a.device)\n    ids = tok.convert_tokens_to_ids([\'<bot>\', \'<eot>\', \'<latent>\']); assert ids == [50257, 50258, 50259] and len(tok) == model.get_input_embeddings().num_embeddings == 50260, (ids, len(tok))   # same special-token ids as in training\n    rows = build_rows(a.gens, a.model, tok, a.limit); R = Runner(tok, model, rows, a.device, a.bs)\n    cnt = {g: sum(r[\'group\'] == g for r in rows) for g in (\'harmful\', \'resistant\', \'wrong_either\')}; print(\'rows\', len(rows), cnt, \'depth known\', sum(r[\'depth\'] is not None for r in rows), flush=True)\n    MH, ZH = R.all(\'h\'); MHn = R.last_norm.copy(); tokdiff = R.tok_diff.copy(); MU, ZU = R.all(\'u\'); MUn = R.last_norm.copy(); MP, ZP = R.all(\'m\')                                       # hinted; natural unhinted; position-matched (hint hidden, positions kept)\n    okH = np.array([(MH[i] < 0) == (rows[i][\'ph\'] == rows[i][\'hint\']) for i in range(len(rows))]); okU = np.array([(MU[i] > 0) == (rows[i][\'pu\'] == rows[i][\'true\']) for i in range(len(rows))])\n    agree, agreeU = float(okH.mean()), float(okU.mean()); gagree = {g: (float(okH[[r[\'group\'] == g for r in rows]].mean()), float(okU[[r[\'group\'] == g for r in rows]].mean())) for g in cnt if cnt[g]}\n    print(f\'rebuild check: sign(M) agrees with the saved prediction on {agree:.3f} (hinted) / {agreeU:.3f} (unhinted); by group (hinted, unhinted): {gagree}\', flush=True)\n    assert min(agree, agreeU) >= a.min_agree, f\'prompt/scoring rebuild disagrees with the saved predictions (< {a.min_agree}): stop and investigate before reading any causal result\'\n    idn, _ = R.all(\'h\', lambda idx: {t: ZH[idx, t] for t in range(N)}); assert np.abs(idn - MH).max() < 1e-3, f\'identity patch differs by {np.abs(idn - MH).max()}\'\n    don = donors(rows, rng); has_don = don >= 0\n    V, base, VN = {}, {}, {}                                                       # V[name] = M under the intervention; base[name] = the baseline it is compared against\n    def add(name, M, b): V[name], base[name], VN[name] = M, b, R.last_norm.copy()      # R.last_norm belongs to the R.all call that produced M (arguments are evaluated before add)\n    if not (a.followup or a.latentdep):\n        for t in range(N):\n            add(f\'single_t{t}\', R.all(\'h\', lambda idx, t=t: {t: ZU[idx, t]})[0], MH)\n            g = torch.Generator().manual_seed(a.seed * 100 + t)\n            add(f\'noise_t{t}\', R.all(\'h\', lambda idx, t=t, g=g: {t: torch.randn(len(idx), ZU.shape[-1], generator=g) * ZU[idx, t].norm(dim=-1, keepdim=True) / np.sqrt(ZU.shape[-1])})[0], MH)\n            add(f\'matched_t{t}\', R.all(\'h\', lambda idx, t=t: {t: ZU[np.where(don[idx] >= 0, don[idx], idx), t]})[0], MH)\n            add(f\'samecond_t{t}\', R.all(\'h\', lambda idx, t=t: {t: ZH[np.where(don[idx] >= 0, don[idx], idx), t]})[0], MH)\n            add(f\'single_pm_t{t}\', R.all(\'h\', lambda idx, t=t: {t: ZP[idx, t]})[0], MH)\n            add(f\'reverse_t{t}\', R.all(\'u\', lambda idx, t=t: {t: ZH[idx, t]})[0], MU)\n            # persist (Amendment 1, secondary): slots 0 and t replaced, slots t+1.. cannot attend to the hint; slot t and earlier cached states can still have seen it\n            add(f\'persist_t{t}\', R.all(\'h\', lambda idx, t=t: {s: ZU[idx, s] for s in {0, t}}, block_latents_from=min(t + 1, N))[0], MH)\n        allU = lambda idx: {t: ZU[idx, t] for t in range(N)}\n        add(\'full6\', R.all(\'h\', allU)[0], MH)\n        # scale / positive controls (Amendment 3): if destroying or swapping ALL latents also leaves M unchanged, the answer is not reading the latents and a null restoration is expected\n        add(\'zero_full6\', R.all(\'h\', lambda idx: {t: torch.zeros_like(ZU[idx, t]) for t in range(N)})[0], MH)\n        add(\'matched_full6\', R.all(\'h\', lambda idx: {t: ZU[np.where(don[idx] >= 0, don[idx], idx), t] for t in range(N)})[0], MH)\n        add(\'block_answer\', R.all(\'h\', block_answer=True)[0], MH)\n        allP = lambda idx: {t: ZP[idx, t] for t in range(N)}\n        add(\'full6_pm\', R.all(\'h\', allP)[0], MH)\n        add(\'block_latents\', R.all(\'h\', lambda idx: {0: ZU[idx, 0]}, block_latents_from=0)[0], MH)                  # slot 0 replaced too: otherwise the hint-contaminated <bot> state leaks in\n        add(\'block_latents_pm\', R.all(\'h\', lambda idx: {0: ZP[idx, 0]}, block_latents_from=0)[0], MH)\n        # matched-attention controls (Amendment 3). (a) disruption baseline: the same blocks on UNHINTED prompts (<bot> blocked, nothing else to block), compared with the unhinted M:\n        # a shift here is caused by the blocking itself, not by removing a hint. Identity patches (own states) keep slot inputs unchanged.\n        allUid = lambda idx: {t: ZU[idx, t] for t in range(N)}\n        add(\'u_block_latents\', R.all(\'u\', lambda idx: {0: ZU[idx, 0]}, block_latents_from=0, block_bot=True)[0], MU)\n        add(\'u_block_answer\', R.all(\'u\', block_answer=True, block_bot=True)[0], MU)\n        add(\'u_both_blocked\', R.all(\'u\', allUid, block_latents_from=0, block_answer=True, block_bot=True)[0], MU)\n        # (b) donor trajectories produced under the SAME attention regime as the blocked recipient (hint hidden, <bot> blocked from every latent step)\n        _, ZB = R.all(\'m\', block_latents_from=0, block_bot=True); allB = lambda idx: {t: ZB[idx, t] for t in range(N)}\n        add(\'block_latents_mad\', R.all(\'h\', lambda idx: {0: ZB[idx, 0]}, block_latents_from=0)[0], MH)\n        add(\'both_blocked_mad\', R.all(\'h\', allB, block_latents_from=0, block_answer=True)[0], MH)\n        add(\'both_blocked_pm\', R.all(\'h\', allP, block_latents_from=0, block_answer=True)[0], MH)\n        add(\'full6_block_answer\', R.all(\'h\', allU, block_answer=True)[0], MH)\n        add(\'both_blocked\', R.all(\'h\', allU, block_latents_from=0, block_answer=True)[0], MH)\n        for fam in (\'single\', \'single_pm\', \'noise\', \'matched\', \'samecond\', \'reverse\', \'persist\'):                        # row-level mean over the six steps (primary = single)\n            add(f\'{fam}_mean\', np.mean([V[f\'{fam}_t{t}\'] for t in range(N)], 0), MH if fam != \'reverse\' else MU); VN[f\'{fam}_mean\'] = np.mean([VN[f\'{fam}_t{t}\'] for t in range(N)], 0)\n    if a.followup:                                                         # follow-up (Amendment 6, exploratory): hint tokens and <bot> blocked separately; none of the registered conditions are recomputed\n        add(\'f_ans_hint\', R.all(\'h\', block_answer=\'hint\')[0], MH)\n        add(\'f_ans_bot\', R.all(\'h\', block_answer=\'bot\')[0], MH)\n        add(\'f_ans_both\', R.all(\'h\', block_answer=True)[0], MH)                           # = block_answer of the main run (consistency check)\n        add(\'f_lat_hint\', R.all(\'h\', block_latents_from=0, latent_kind=\'hint\')[0], MH)\n        add(\'f_lat_bot\', R.all(\'h\', block_latents_from=0, latent_kind=\'bot\')[0], MH)\n        add(\'f_lat_hint_ans_hint\', R.all(\'h\', block_latents_from=0, latent_kind=\'hint\', block_answer=\'hint\')[0], MH)\n        add(\'f_lat_bot_ans_bot\', R.all(\'h\', block_latents_from=0, latent_kind=\'bot\', block_answer=\'bot\')[0], MH)\n        add(\'f_u_ans_bot\', R.all(\'u\', block_answer=\'bot\', block_bot=True)[0], MU)           # disruption baseline: blocking <bot> alone on UNHINTED prompts\n        add(\'f_u_lat_bot_ans_bot\', R.all(\'u\', block_latents_from=0, latent_kind=\'bot\', block_answer=\'bot\', block_bot=True)[0], MU)\n    if a.latentdep:                                                        # diagnostic (Amendment 7): do the latents matter for the plain, unhinted answer?\n        add(\'u_zero_full6\', R.all(\'u\', lambda idx: {t: torch.zeros_like(ZU[idx, t]) for t in range(N)})[0], MU)\n        add(\'u_matched_full6\', R.all(\'u\', lambda idx: {t: ZU[np.where(don[idx] >= 0, don[idx], idx), t] for t in range(N)})[0], MU)\n        for t in range(N): add(f\'u_zero_t{t}\', R.all(\'u\', lambda idx, t=t: {t: torch.zeros_like(ZU[idx, t])})[0], MU)\n    D = {k: V[k] - base[k] for k in V}                                                                 # change in M (positive = towards the true option)\n    cl = [r[\'q\'] for r in rows]; S = {}\n    for g in cnt:\n        m = np.array([r[\'group\'] == g for r in rows])\n        if not m.any(): continue\n        S[g] = {}\n        for k in V:\n            mm = m & (has_don if k.startswith((\'matched\', \'samecond\')) else True)\n            if not mm.any(): continue\n            d = D[k][mm]; est = boot(d, [c for c, x in zip(cl, mm) if x], a.boot, rng)\n            S[g][k] = dict(n=int(mm.sum()), mean_dM=est[0], lo=est[1], hi=est[2], p_true=float((V[k][mm] > 0).mean()), p_flip_to_wrong=float((V[k][mm] < 0).mean()))\n        LEN = (\'single_mean\', \'single_pm_mean\', \'block_answer\', \'block_latents\', \'both_blocked\', \'both_blocked_pm\', \'both_blocked_mad\', \'full6\')           # length sensitivity (M_norm = per-option-token mean log-prob margin; M stays primary)\n        dn = lambda k: VN[k] - (MUn if base[k] is MU else MHn)\n        S[g][\'_length_sensitivity\'] = dict(\n            option_token_count_differs=float((tokdiff[m] != 0).mean()), mean_abs_token_diff=float(np.abs(tokdiff[m]).mean()),\n            sign_disagree_M_vs_Mnorm_hinted=float((np.sign(MH[m]) != np.sign(MHn[m])).mean()), sign_disagree_M_vs_Mnorm_unhinted=float((np.sign(MU[m]) != np.sign(MUn[m])).mean()),\n            **{k: dict(mean_dMnorm=boot(dn(k)[m], [c for c, x in zip(cl, m) if x], a.boot, rng)[0], sign_disagree_M_vs_Mnorm=float((np.sign(V[k][m]) != np.sign(VN[k][m])).mean())) for k in LEN if k in V})\n        S[g][\'_baseline\'] = dict(M_hinted=float(MH[m].mean()), M_unhinted=float(MU[m].mean()), p_true_hinted=float((MH[m] > 0).mean()))\n    DESC = {\'single_t{t}\': \'hinted run; the INPUT vector of latent slot t replaced by the same question\\\'s natural UNHINTED state (shorter prompt, so shifted absolute positions); later slots and the answer recomputed; slots can still attend to the hinted prompt\',\n            \'single_pm_t{t}\': \'as single_t but the donor is the position-matched run (hinted prompt, hint tokens hidden from attention, same position ids)\',\n            \'noise_t{t}\': \'slot t input replaced by Gaussian noise with the norm of the unhinted donor state\',\n            \'matched_t{t}\': \'slot t input replaced by the unhinted state of ANOTHER question (same group and depth where known)\',\n            \'samecond_t{t}\': \'slot t input replaced by the hinted state of another question of the same group\',\n            \'reverse_t{t}\': \'UNHINTED run; slot t input replaced by the same question\\\'s hinted state (change relative to the unhinted M)\',\n            \'persist_t{t}\': \'slots 0 and t replaced by unhinted states; slots t+1.. cannot attend to the hint span or <bot>; slot t and earlier cached states may have seen the hint (secondary)\',\n            \'*_mean\': \'row-level mean of the six per-step conditions of that family (single_mean is the registered primary)\',\n            \'full6\': \'all six slot inputs replaced by natural unhinted states; the hinted prompt stays accessible to the slots and to the answer\',\n            \'zero_full6\': \'all six slot inputs replaced by zeros (scale control); hinted prompt stays accessible\',\n            \'matched_full6\': \'all six slot inputs replaced by another question\\\'s unhinted states (scale control); hinted prompt stays accessible\',\n            \'full6_pm\': \'all six slot inputs replaced by position-matched states; hinted prompt stays accessible\',\n            \'block_answer\': \'answer positions cannot attend to the hint span or <bot>; the latents run unchanged and may carry the hint\',\n            \'block_latents\': \'slot 0 input replaced by the unhinted state, and latent steps cannot attend to the hint span or <bot>; the answer can still read the hint directly\',\n            \'block_latents_pm\': \'as block_latents with the position-matched slot-0 state\', \'block_latents_mad\': \'as block_latents with slot 0 from the matched-attention donor run (hint hidden, <bot> blocked)\',\n            \'full6_block_answer\': \'all six inputs replaced by unhinted states and the answer blocked from the hint span and <bot>; slots can still attend to the hint\',\n            \'both_blocked\': \'all six inputs replaced by unhinted states, latents AND answer blocked from the hint span and <bot>: no route from the hint remains\',\n            \'both_blocked_pm\': \'as both_blocked with position-matched donor states\', \'both_blocked_mad\': \'as both_blocked with donor states produced under the same blocked-attention regime\',\n            \'f_ans_hint / f_ans_bot / f_ans_both\': \'FOLLOW-UP: the answer cannot attend to the hint tokens only / to <bot> only / to both (= block_answer); latents untouched\',\n            \'f_lat_hint / f_lat_bot\': \'FOLLOW-UP: latent steps (from slot 0) cannot attend to the hint tokens only / to <bot> only; slot-0 input is NOT replaced; the answer reads everything\',\n            \'f_lat_hint_ans_hint / f_lat_bot_ans_bot\': \'FOLLOW-UP: the same single-target block applied to both latent steps and the answer\',\n            \'f_u_ans_bot / f_u_lat_bot_ans_bot\': \'FOLLOW-UP disruption baselines: <bot> alone blocked on UNHINTED prompts; change relative to the unhinted M\',\n            \'u_zero_full6 / u_matched_full6 / u_zero_t*\': \'DIAGNOSTIC on the UNHINTED prompt: all six latent inputs zeroed / replaced by another question\\\'s unhinted states / one slot zeroed; change relative to the unhinted M\',\n            \'u_block_latents / u_block_answer / u_both_blocked\': \'disruption baselines: the same blocks applied to UNHINTED prompts (<bot> blocked); change relative to the unhinted M, i.e. what blocking does with no hint present\'}\n    meta = dict(latentdep=a.latentdep, followup=a.followup, third_option_rows_skipped=build_rows.third_option_skipped, run_id=a.run_id, ckpt=a.ckpt, model=a.model, n_rows=len(rows), counts=cnt, depth_known=int(sum(r[\'depth\'] is not None for r in rows)), rebuild_agreement_hinted=agree, rebuild_agreement_unhinted=agreeU, rebuild_agreement_by_group=gagree,\n                identity_max_abs_diff=float(np.abs(idn - MH).max()), seed=a.seed, boot=a.boot, minutes=(time.time() - t0) / 60, torch=torch.__version__)\n    json.dump(dict(meta=meta, condition_descriptions=DESC, summary=S), open(f\'{a.out}/patching_summary.json\', \'w\'), indent=1)\n    with open(f\'{a.out}/patching_rows.jsonl\', \'w\') as f:\n        for i, r in enumerate(rows): f.write(json.dumps(dict(q=r[\'q\'], group=r[\'group\'], M_hinted=MH[i], M_unhinted=MU[i], **{k: float(V[k][i]) for k in V})) + \'\\n\')\n    h = S.get(\'harmful\', {})\n    print(f"\\nPRIMARY (harmful switches, single-step same-question restoration, mean over steps): dM = {h[\'single_mean\'][\'mean_dM\']:.3f} [{h[\'single_mean\'][\'lo\']:.3f}, {h[\'single_mean\'][\'hi\']:.3f}], n = {h[\'single_mean\'][\'n\']}" if \'single_mean\' in h else \'no harmful rows\')\n    for g in S:\n        print(f\'\\n[{g}] n={cnt[g]}  baseline {S[g]["_baseline"]}\')\n        for k in (\'u_zero_full6\', \'u_matched_full6\', \'u_zero_t0\', \'u_zero_t1\', \'u_zero_t2\', \'u_zero_t3\', \'u_zero_t4\', \'u_zero_t5\', \'f_ans_hint\', \'f_ans_bot\', \'f_ans_both\', \'f_lat_hint\', \'f_lat_bot\', \'f_lat_hint_ans_hint\', \'f_lat_bot_ans_bot\', \'f_u_ans_bot\', \'f_u_lat_bot_ans_bot\', \'single_mean\', \'single_pm_mean\', \'noise_mean\', \'matched_mean\', \'samecond_mean\', \'persist_mean\', \'full6\', \'zero_full6\', \'matched_full6\', \'full6_pm\', \'block_answer\', \'block_latents\', \'block_latents_pm\', \'full6_block_answer\', \'both_blocked\', \'both_blocked_pm\', \'block_latents_mad\', \'both_blocked_mad\', \'u_block_latents\', \'u_block_answer\', \'u_both_blocked\', \'reverse_mean\'):\n            if k in S[g]: e = S[g][k]; print(f\'  {k:20s} dM {e["mean_dM"]:+.3f} [{e["lo"]:+.3f},{e["hi"]:+.3f}]  P(true) {e["p_true"]:.2f}  P(wrong) {e["p_flip_to_wrong"]:.2f}\')\n    print(f\'\\nsaved to {a.out} in {meta["minutes"]:.1f} min\')\n\n\nif __name__ == \'__main__\': main()\n')
os.makedirs('/kaggle/working/tests', exist_ok=True)
with open('/kaggle/working/tests/test_latent_patching.py', 'w') as f: f.write('"""Validation of analysis/latent_patching.py on a tiny random GPT-2 (protocol Amendment 1, item 2). Run: python -m pytest prosqa_monitor/tests/test_latent_patching.py"""\nimport os, sys, unittest\nimport torch\nfrom transformers import GPT2Config, GPT2LMHeadModel\nsys.path.insert(0, \'/kaggle/working\')\nimport latent_patching as lp\n\nPAD, BOT, EOT, LAT, N = 0, 1, 2, 3, 4\n\n\ndef tiny():\n    torch.manual_seed(0)\n    m = GPT2Config(vocab_size=40, n_positions=64, n_embd=32, n_layer=2, n_head=2, resid_pdrop=0.1, embd_pdrop=0.1, attn_pdrop=0.1)\n    return GPT2LMHeadModel(m).eval()\n\n\nPROMPTS = [[5, 6, 7, 8, 9, 10, 11, 12], [13, 14, 15, 16, 17]]       # different lengths -> left padding\nHINTS = [3, 2]                                                      # trailing hint-line tokens\nTAILS = [[EOT, 20, 21, 22], [EOT, 23, 24]]\n\n\ndef batch(prompts=PROMPTS, hints=HINTS, tails=TAILS):\n    return lp.make_batch(prompts, hints, tails, N, PAD, BOT, LAT)\n\n\nclass T(unittest.TestCase):\n    def setUp(self): self.m = tiny()\n\n    def test_identity_patch(self):\n        bt = batch(); base = lp.run(self.m, bt)\n        pat = lp.run(self.m, bt, patch={i: base[\'thoughts\'][:, i] for i in range(N)})\n        self.assertTrue(torch.allclose(base[\'logp\'], pat[\'logp\'], atol=1e-6))\n        self.assertTrue(torch.allclose(base[\'thoughts\'], pat[\'thoughts\'], atol=1e-6))\n\n    def test_unhinted_to_unhinted_changes_nothing(self):\n        un = [p[:len(p) - h] for p, h in zip(PROMPTS, HINTS)]\n        bt = batch(un, [0, 0]); a = lp.run(self.m, bt)\n        b = lp.run(self.m, bt, patch={2: a[\'thoughts\'][:, 2]})\n        self.assertTrue(torch.allclose(a[\'logp\'], b[\'logp\'], atol=1e-6))\n\n    def test_only_intended_latent_replaced(self):\n        bt = batch(); base = lp.run(self.m, bt); donor = torch.randn(2, 32)\n        out = lp.run(self.m, bt, patch={2: donor})\n        self.assertTrue(torch.allclose(out[\'thoughts\'][:, 2], donor))                     # the patched slot\'s input is the donor\n        self.assertTrue(torch.allclose(out[\'thoughts\'][:, :2], base[\'thoughts\'][:, :2]))   # earlier slots untouched\n        self.assertFalse(torch.allclose(out[\'thoughts\'][:, 3], base[\'thoughts\'][:, 3]))    # later slot recomputed\n\n    def test_downstream_recomputed(self):\n        bt = batch(); base = lp.run(self.m, bt); out = lp.run(self.m, bt, patch={0: torch.randn(2, 32)})\n        for i in range(1, N): self.assertFalse(torch.allclose(out[\'thoughts\'][:, i], base[\'thoughts\'][:, i]))\n        self.assertFalse(torch.allclose(out[\'logp\'], base[\'logp\']))\n\n    def test_eval_no_grad_deterministic(self):\n        bt = batch(); a, b = lp.run(self.m, bt), lp.run(self.m, bt)\n        self.assertTrue(torch.equal(a[\'logp\'], b[\'logp\']))\n        self.m.train()\n        with self.assertRaises(AssertionError): lp.run(self.m, bt)\n        self.assertFalse(a[\'logp\'].requires_grad)\n\n    def test_hint_span_masking(self):\n        bt = batch(); a = bt[\'a\']\n        for b, h in enumerate(HINTS):\n            idx = bt[\'block\'][b].nonzero().flatten().tolist()\n            self.assertEqual(idx, list(range(a - 1 - h, a)))                              # hint tokens + <bot>, nothing else\n            self.assertEqual(bt[\'ids\'][b, a - 1].item(), BOT)\n            self.assertEqual(bt[\'ids\'][b, a - 1 - h:a - 1].tolist(), PROMPTS[b][-h:])\n        base = lp.run(self.m, bt)\n        self.assertFalse(torch.allclose(base[\'logp\'], lp.run(self.m, bt, block_answer=True)[\'logp\']))\n        self.assertFalse(torch.allclose(base[\'logp\'], lp.run(self.m, bt, block_latents_from=0)[\'logp\']))\n        # blocking from a slot after the last one changes nothing\n        self.assertTrue(torch.allclose(base[\'logp\'], lp.run(self.m, bt, block_latents_from=N)[\'logp\']))\n\n    def test_hint_span_len(self):\n        enc = lambda s: [ord(c) % 30 + 5 for c in s]\n        self.assertEqual(lp.hint_span_len(enc, \'abc\\nHint: x\', \'abc\\n\'), 7)\n        with self.assertRaises(AssertionError): lp.hint_span_len(lambda s: [len(s)] + enc(s)[1:], \'abc\\nH\', \'abc\\n\')\n\n    def test_multitoken_scoring_matches_direct_forward(self):\n        bt = batch(); out = lp.run(self.m, bt); a = bt[\'a\']\n        emb = self.m.get_input_embeddings()(bt[\'ids\'])\n        full = torch.cat([emb[:, :a], out[\'thoughts\'], emb[:, a + N:]], 1)                # latent slot inputs = the recorded thoughts\n        att, pos = bt[\'att\'], bt[\'pos\']\n        logits = self.m(inputs_embeds=full, attention_mask=att, position_ids=pos).logits[:, a + N:]\n        ref = torch.log_softmax(logits[:, :-1], -1).gather(-1, bt[\'tail_ids\'][:, 1:, None])[..., 0]\n        self.assertTrue(torch.allclose((ref * out[\'mask\']).sum(1), lp.sentence_logprob(out), atol=1e-4))\n\n    def test_padding_invariance(self):\n        both = lp.sentence_logprob(lp.run(self.m, batch()))\n        solo = lp.sentence_logprob(lp.run(self.m, batch(PROMPTS[1:], HINTS[1:], TAILS[1:])))\n        self.assertTrue(torch.allclose(both[1:], solo, atol=1e-4))\n\n    def test_margin_sign_and_consistency(self):\n        tt, tw = [[EOT, 20, 21], [EOT, 20, 21]], [[EOT, 25, 26, 27], [EOT, 25, 26, 27]]\n        M, _ = lp.margin(self.m, PROMPTS, HINTS, tt, tw, N, PAD, BOT, LAT)\n        s_t = lp.sentence_logprob(lp.run(self.m, batch(tails=tt))); s_w = lp.sentence_logprob(lp.run(self.m, batch(tails=tw)))\n        self.assertTrue(torch.allclose(M, s_t - s_w))\n\n\n    def test_zero_hint_blocks_nothing(self):\n        bt = batch(PROMPTS, [0, 0]); self.assertFalse(bt[\'block\'].any())\n\n    def test_hide_hint_keeps_positions_removes_content(self):\n        hid = batch(); hid2 = lp.make_batch(PROMPTS, HINTS, TAILS, N, PAD, BOT, LAT, hide_hint=True); a = hid[\'a\']\n        self.assertTrue(torch.equal(hid[\'pos\'], hid2[\'pos\']))                              # position ids identical to the hinted run\n        for b, h in enumerate(HINTS):\n            self.assertEqual(hid2[\'att\'][b, a - 1 - h:a - 1].sum().item(), 0)               # hint tokens hidden\n            self.assertEqual(hid2[\'att\'][b, a - 1].item(), 1)                               # <bot> still visible\n        other = [p[:-h] + [30 + j for j in range(h)] for p, h in zip(PROMPTS, HINTS)]       # different hint content, same length\n        o1 = lp.run(self.m, hid2); o2 = lp.run(self.m, lp.make_batch(other, HINTS, TAILS, N, PAD, BOT, LAT, hide_hint=True))\n        self.assertTrue(torch.allclose(o1[\'logp\'], o2[\'logp\'], atol=1e-5))                  # hidden hint content cannot matter\n\n    def test_blocking_needs_slot0_replacement(self):\n        other = [p[:-h] + [30 + j for j in range(h)] for p, h in zip(PROMPTS, HINTS)]\n        a, b = batch(), lp.make_batch(other, HINTS, TAILS, N, PAD, BOT, LAT)\n        z0 = torch.randn(2, 32)\n        leak = lambda bt: lp.run(self.m, bt, block_latents_from=0)[\'thoughts\']\n        self.assertFalse(torch.allclose(leak(a), leak(b)))                                   # masking alone leaks the hint through slot 0\'s input\n        held = lambda bt: lp.run(self.m, bt, patch={0: z0}, block_latents_from=0)[\'thoughts\']\n        self.assertTrue(torch.allclose(held(a), held(b), atol=1e-5))                         # with slot 0 replaced, the latents cannot see the hint\n\n    def test_both_blocked_is_hint_independent(self):\n        other = [p[:-h] + [30 + j for j in range(h)] for p, h in zip(PROMPTS, HINTS)]\n        z = {i: torch.randn(2, 32) for i in range(N)}\n        f = lambda ps: lp.run(self.m, lp.make_batch(ps, HINTS, TAILS, N, PAD, BOT, LAT), patch=z, block_latents_from=0, block_answer=True)[\'logp\']\n        self.assertTrue(torch.allclose(f(PROMPTS), f(other), atol=1e-5))\n        g = lambda ps: lp.run(self.m, lp.make_batch(ps, HINTS, TAILS, N, PAD, BOT, LAT), patch=z)[\'logp\']\n        self.assertFalse(torch.allclose(g(PROMPTS), g(other)))                              # without the blocks the hint still reaches the answer\n\n    def test_block_bot_on_unhinted(self):\n        bt = lp.make_batch(PROMPTS, [0, 0], TAILS, N, PAD, BOT, LAT, block_bot=True); a = bt[\'a\']\n        self.assertEqual(bt[\'block\'].sum().item(), 2); self.assertTrue(bt[\'block\'][:, a - 1].all())\n        base = lp.run(self.m, bt)\n        self.assertFalse(torch.allclose(base[\'logp\'], lp.run(self.m, bt, block_answer=True)[\'logp\']))      # blocking <bot> alone disturbs the model: the disruption baseline is non-zero\n\n    def test_option_only_normalised_margin(self):\n        tt, tw = [[EOT, 20, 21, 22, 23, 9]] * 2, [[EOT, 20, 21, 25, 9]] * 2               # shared prefix 20,21; option words (22,23) and (25,); terminal period token 9 excluded\n        M, _, Mn, dn = lp.margin(self.m, PROMPTS, HINTS, tt, tw, N, PAD, BOT, LAT, opt_start=[2, 2])\n        ot, ow = lp.run(self.m, batch(tails=tt)), lp.run(self.m, batch(tails=tw))\n        exp = (ot[\'logp\'][:, 2:4].sum(1) / 2) - (ow[\'logp\'][:, 2:3].sum(1) / 1)\n        self.assertTrue(torch.allclose(Mn, exp, atol=1e-5)); self.assertEqual(dn.tolist(), [1, 1])\n        self.assertTrue(torch.allclose(M, lp.sentence_logprob(ot) - lp.sentence_logprob(ow), atol=1e-5))   # summed M is unchanged by the extra argument\n\n    def test_separate_hint_and_bot_blocking(self):\n        bt = batch(); a = bt[\'a\']\n        for b, h in enumerate(HINTS):\n            self.assertEqual(bt[\'block_hint\'][b].nonzero().flatten().tolist(), list(range(a - 1 - h, a - 1)))      # hint tokens only\n            self.assertEqual(bt[\'block_botpos\'][b].nonzero().flatten().tolist(), [a - 1])                          # <bot> only\n        self.assertTrue(torch.equal(bt[\'block\'], bt[\'block_hint\'] | bt[\'block_botpos\']))                          # the union is the old mask\n        unh = lp.make_batch(PROMPTS, [0, 0], TAILS, N, PAD, BOT, LAT)\n        self.assertFalse(unh[\'block_hint\'].any()); self.assertFalse(unh[\'block_botpos\'].any())                    # nothing to block without block_bot\n        self.assertTrue(lp.make_batch(PROMPTS, [0, 0], TAILS, N, PAD, BOT, LAT, block_bot=True)[\'block_botpos\'][:, a - 1].all())\n        r = {k: lp.run(self.m, bt, block_answer=k)[\'logp\'] for k in (True, \'hint\', \'bot\')}\n        self.assertFalse(torch.allclose(r[\'hint\'], r[\'bot\'])); self.assertFalse(torch.allclose(r[True], r[\'hint\'])); self.assertFalse(torch.allclose(r[True], r[\'bot\']))\n        self.assertTrue(torch.allclose(r[True], lp.run(self.m, bt, block_answer=True, latent_kind=\'hint\')[\'logp\']))   # latent_kind only matters for blocked latent steps\n        l = {k: lp.run(self.m, bt, block_latents_from=0, latent_kind=k)[\'thoughts\'] for k in (\'both\', \'hint\', \'bot\')}\n        self.assertFalse(torch.allclose(l[\'hint\'], l[\'bot\']))\n        self.assertTrue(torch.allclose(lp.run(self.m, bt, block_latents_from=0)[\'logp\'], lp.run(self.m, bt, block_latents_from=0, latent_kind=\'both\')[\'logp\']))\n\n\nif __name__ == \'__main__\': unittest.main()\n')

In [ ]:
r = subprocess.run([sys.executable, '/kaggle/working/tests/test_latent_patching.py'], capture_output=True, text=True); print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, 'unit tests failed on this transformers/torch version: stop'

In [ ]:
# two workers, one per GPU: GPU 0 takes seeds 0 and 2, GPU 1 takes seed 1 (seeds are independent; each worker sees a single device)
LIMIT, BOOT, PFX, FOLLOWUP = 0, 2000, 'patching', False
SEEDS = [0, 1]
NG = max(torch.cuda.device_count(), 1)
PLAN = {0: [0, 2], 1: [1]} if (SEEDS == [0, 1, 2] and NG >= 2) else {g: SEEDS[g::NG] for g in range(NG) if SEEDS[g::NG]}
RUN_ID = str(int(time.time()))
import shutil
DIRS = [('latentdep', ' --latentdep'), ('patching', '')]
for s in SEEDS:
    for d, _ in DIRS: shutil.rmtree(f'/kaggle/working/{d}_s{s}', ignore_errors=True)
def worker(gpu, seeds):
    cmds = ' && '.join(f"{sys.executable} /kaggle/working/run_patching.py --ckpt {INP}/s{s}_coconut_b.pt --gens {INP}/s{s}_generations.jsonl --out /kaggle/working/{d}_s{s} --device cuda --seed {s} --boot {BOOT} --bs 32 --run-id {RUN_ID}{fl}" + (f' --limit {LIMIT}' if LIMIT else '') + f' > /kaggle/working/{d}_s{s}.log 2>&1' for s in seeds for d, fl in DIRS)
    return subprocess.Popen(['bash', '-c', cmds], env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'PROSQA_DIR': DATA})
t0 = time.time(); procs = [worker(g, s) for g, s in PLAN.items()]
for p in procs: p.wait()
print('workers exited', [p.returncode for p in procs], f'{(time.time() - t0) / 60:.1f} min')

In [ ]:
for s in SEEDS:
  for d, _ in DIRS:
    f = f'/kaggle/working/{d}_s{s}.log'
    if os.path.exists(f): print(f'===== {d} seed {s} =====\n' + ''.join(l for l in open(f) if 'arn' not in l)[-5000:])

In [ ]:
assert all(p.returncode == 0 for p in procs), 'a worker failed: see the logs above'
for s in SEEDS:
  for d, _ in DIRS:
    f = f'/kaggle/working/{d}_s{s}/patching_summary.json'; assert os.path.exists(f), f'missing {f}'
    assert json.load(open(f))['meta']['run_id'] == RUN_ID, f'{d} seed {s} summary is not from this run'
print('all requested summaries present and from this run', RUN_ID, SEEDS)